## Import libraries

In [1]:
import os
import sys
import tensorflow                             as tf
import numpy                                  as np

import shutil
import h5py
import seaborn                                as sns
import pandas                                 as pd
import matplotlib.pyplot                      as plt
from sklearn.cluster                      import KMeans
from tqdm.notebook                        import tqdm

In [2]:
import matplotlib as mpl
from matplotlib import pyplot as plt
%matplotlib notebook

# Use custom style
# See: https://matplotlib.org/1.5.3/users/style_sheets.html
style_name = 'paper_1column'
mpl.style.use(
    os.environ['WORKSPACE_PATH'] \
        + '/RESEARCH/styles/matplotlib/' \
        + style_name + '.mplstyle'
)
# Get colors list
COLORS = mpl.rcParams['axes.prop_cycle'].by_key()['color']*10

## Define extra inputs

In [3]:
# Paths
WORKSPACE_PATH    = os.getenv('WORKSPACE_PATH')  
SurQCTFldr        = WORKSPACE_PATH + '/ML/SurQCT/surqct/'
PathToRunFld      = SurQCTFldr + f'/../KInel/Run_9/'
OutputFldr        = './Outputs'

# Branch/Trunk layer index
OutBranchLayerIdx = 17
OutTrunkLayerIdx  = 18

# Misc
Eh_to_eV          = 27.211396641308
NBins             = [3,6,10]
EDiss             = 5.115

In [4]:
# Creating output folder
try:
    shutil.rmtree(OutputFldr)
except:
    pass
os.makedirs(OutputFldr)

## Initialize SurQCT model

In [5]:
print('\n[SurQCT]: Loading Input Module ...\n')

sys.path.insert(0, PathToRunFld)
from SurQCT_Input import inputdata

sys.path.insert(0, SurQCTFldr  + '/src/Reading/')
from Reading import read_levelsdata


[SurQCT]: Loading Input Module ...



In [6]:
print('\n[SurQCT]: Initializing Input ...\n')

InputData = inputdata(WORKSPACE_PATH, SurQCTFldr)


[SurQCT]: Initializing Input ...



In [7]:
print('\n[SurQCT]: Loading Model Module ...\n')

sys.path.insert(0, SurQCTFldr  + '/src/Model/' + InputData.ApproxModel + '/')
from Model import model


[SurQCT]: Loading Model Module ...



In [8]:
print('\n[SurQCT]: Initializing SurQCT Model and Loading its Parameters ...\n')

SurQCTModel = model(InputData, PathToRunFld, None, None)
SurQCTModel.load_params(InputData.PathToParamsFld)


[SurQCT]: Initializing SurQCT Model and Loading its Parameters ...

[SurQCT]:   Variables Selected for Training:
[SurQCT]:     x =  ['log_EVib_i', 'log_ERot_i', 'ri_i', 'log_rorMin_i', 'TTran_i', 'log_EVib_Delta', 'log_ERot_Delta', 'ri_Delta', 'log_rorMin_Delta', 'TTran_Delta']
[SurQCT]:     y =  KExcit
[SurQCT]:   Loading ML Model from Folder: /home/zanardi/WORKSPACE//ML/SurQCT/surqct//../KInel/Run_9//MyModel
[SurQCT]:   Restoring ML Model Weigths to File: /home/zanardi/WORKSPACE//ML/SurQCT/surqct//../KInel/Run_9//MyModel/MyWeights.h5
[SurQCT]:   Loading ML Model Parameters from File:  /home/zanardi/WORKSPACE//ML/SurQCT/surqct//../KInel/Run_9//Params//ModelCheckpoint/cp-1963.ckpt


In [9]:
print('\n[SurQCT]: Compiling Branch and Trunk networks ...\n')

# Branch ----------------------------------------------------------------------
if OutBranchLayerIdx is None:
    raise ValueError
Branch = tf.keras.Model(
    SurQCTModel.Model.inputs,
    SurQCTModel.Model.layers[OutBranchLayerIdx].output,
    name='Branch'
)
Branch.summary()

# Trunk -----------------------------------------------------------------------
if OutTrunkLayerIdx is None:
    raise ValueError
Trunk = tf.keras.Model(
    SurQCTModel.Model.inputs,
    SurQCTModel.Model.layers[OutTrunkLayerIdx].output,
    name='Trunk'
)
Trunk.summary()


[SurQCT]: Compiling Branch and Trunk networks ...

Model: "Branch"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_1 (InputLayer)         [(None, 10)]              0         
_________________________________________________________________
tf.split (TFOpLambda)        [(None, 5), (None, 5)]    0         
_________________________________________________________________
normalization (Normalization (None, 5)                 11        
_________________________________________________________________
Branch_HL1_i (Dense)         (None, 64)                384       
_________________________________________________________________
dropout (Dropout)            (None, 64)                0         
_________________________________________________________________
softmax (Softmax)            (None, 64)                0         
_________________________________________________________________
Branch_H

# Clustering
Cluster levels by applying k-means technique at the branch output

In [10]:
ClusterOutputFldr = OutputFldr + '/Clustering/'
try:
    os.makedirs(ClusterOutputFldr)
except:
    pass

In [11]:
print('\n[SurQCT]: Retrieving diatomic potential data ...\n')

LevelsData = pd.read_csv(
    InputData.PathToDiatFile[-1], delim_whitespace=True, skiprows=15, header=None
)
LevelsData.columns = [
    'vqn','jqn','eint','egam','rmin','rmax','vmin','vmax',' tau','ri','ro'
]
for k in ['eint', 'vmin', 'vmax']:
    LevelsData[k] = LevelsData[k] * Eh_to_eV
LevelsData['eint'] = LevelsData['eint'] - LevelsData['eint'][0] - EDiss


[SurQCT]: Retrieving diatomic potential data ...



In [12]:
print('\n[SurQCT]: Composing input from levels data ...\n')

InpLevelsData = pd.read_csv(
    InputData.PathToLevelsFile[0], header=0, skiprows=0
)[InputData.xVarsVec_i]


[SurQCT]: Composing input from levels data ...



In [13]:
print('\n[SurQCT]: Perfoming branch inference for each TTran ...\n')

TTran = InputData.TTranVecTest
if not isinstance(TTran, np.ndarray):
    TTran = tf.nest.flatten(TTran)

BranchPred = {}
for Ti in TTran:
    
    # Construct input data
    i = InpLevelsData
    T = np.ones((len(i),1))*Ti
    j = np.zeros((len(i),4))
    x = np.hstack([i,T,j,T])

    # Perform inference
    y = Branch.predict(x)
    ColumnNames = [f'b{i+1}' for i in range(y.shape[1])]
    
    # Compose output data
    BranchPred[Ti] = pd.concat(
        [LevelsData, pd.DataFrame(y, columns=ColumnNames)], axis=1
    )
    BranchPred[Ti].to_csv(
        ClusterOutputFldr + f'/Branch_T{Ti}.csv',
        index=False
    )


[SurQCT]: Perfoming branch inference for each TTran ...



In [14]:
print('\n[SurQCT]: K-means clustering ...\n')

LevToBin = {}
NBins = tf.nest.flatten(NBins)
for n in NBins:
    LevToBin[n] = {}
    init = 'k-means++'
    for Ti in TTran:
        X = BranchPred[Ti][ColumnNames].to_numpy()
        res = KMeans(n, init=init).fit(X)
        LevToBin[n][Ti] = res.labels_ + 1
        init = res.cluster_centers_
    pd.DataFrame.from_dict(LevToBin[n], dtype=int).to_csv(
        ClusterOutputFldr + f'/mapping_{n}g.csv',
        index=False
    )


[SurQCT]: K-means clustering ...



/home/zanardi/.local/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:984: RuntimeWarning: Explicit initial center position passed: performing only one init in KMeans instead of n_init=10.
  self._check_params(X)
/home/zanardi/.local/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:984: RuntimeWarning: Explicit initial center position passed: performing only one init in KMeans instead of n_init=10.
  self._check_params(X)
/home/zanardi/.local/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:984: RuntimeWarning: Explicit initial center position passed: performing only one init in KMeans instead of n_init=10.
  self._check_params(X)
/home/zanardi/.local/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:984: RuntimeWarning: Explicit initial center position passed: performing only one init in KMeans instead of n_init=10.
  self._check_params(X)
/home/zanardi/.local/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:984: RuntimeWarning: Explicit initial center pos

In [15]:
def plot_clusters(
    TTran,
    NBins,
    x,
    y,
    xticks,
    yticks,
    xlabel,
    ylabel,
    fig_name
):
    # Set x- and y-axis -----------------------------------------------------------
    # Limits
    xlim = [x.min(), x.max()]
    ylim = [y.min(), y.max()]

    # Initilize subplots ----------------------------------------------------------
    iMax, jMax = len(TTran), len(NBins)
    fig, axs = plt.subplots(
        iMax, jMax, sharey=True, sharex=True, figsize=np.array([6.4,4.8])*2.5
    )
    axs = axs.reshape(iMax, jMax)

    # Subplots labeling -----------------------------------------------------------
    cols = [f'{n} bins' for n in NBins]
    rows = [f'{int(Ti)} K' for Ti in TTran]
    # Column-wise
    for ax, col in zip(axs[0], cols):
        ax.set_title(col, pad=30)
    # Row-wise
    for ax, row in zip(axs[:,-1], rows):
        ax.yaxis.set_label_position("right")
        ax.set_ylabel(row, rotation=-90, labelpad=60)

    # Plot clusters ---------------------------------------------------------------
    for i in range(iMax):
        for j in range(jMax):
            data = LevToBin[NBins[j]][TTran[i]]
            axs[i,j].scatter(x, y, c=data, s=20)
            axs[i,j].set_xlim(xlim)
            axs[i,j].set_ylim(ylim)
            axs[i,j].set_xticks(xticks)
            axs[i,j].set_yticks(yticks)
    fig.add_subplot(111, frame_on=False)
    plt.tick_params(labelcolor="none", bottom=False, left=False)
    plt.xticks([])
    plt.yticks([])
    plt.xlabel(xlabel, labelpad=40)
    plt.ylabel(ylabel, labelpad=70)

    fig.savefig(ClusterOutputFldr + fig_name, dpi=300)

In [16]:
print('\n[SurQCT]: Plot clusters ...\n')

plot_clusters(
    TTran,
    NBins,
    x=LevelsData['vqn'],
    y=LevelsData['jqn'],
    xticks=[LevelsData['vqn'].min(),15,30,LevelsData['vqn'].max()],
    yticks=[LevelsData['jqn'].min(),50,100,150,200,LevelsData['jqn'].max()],
    xlabel='\\textbf{v}',
    ylabel='\\textbf{J}',
    fig_name='/vjClusters.pdf'
)

plot_clusters(
    TTran,
    NBins,
    x=LevelsData['jqn'],
    y=LevelsData['eint'],
    xticks=[LevelsData['jqn'].min(),50,100,150,200,LevelsData['jqn'].max()],
    yticks=[-5.0, -2.5, 0., 2.5],
    xlabel='\\textbf{J}',
    ylabel='\\textbf{E$_i$ [eV]}',
    fig_name='/jeClusters.pdf'
)


[SurQCT]: Plot clusters ...



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Modes reconstruction

In [23]:
gridPts = [100, 100, 3, 3, 3]
TTran = np.array([1000.,10000.,20000.])

ModesOutputFldr = OutputFldr + '/Modes/'
try:
    os.makedirs(ModesOutputFldr)
except:
    pass

In [24]:
# Get mean and variance of trunk inputs
TrunkNormLay = SurQCTModel.Model.layers[4]
mean, std = TrunkNormLay.mean.numpy(), np.sqrt(TrunkNormLay.variance.numpy())

In [25]:
# Compose trunk inputs as 1D arrays
InpArrays = []
for i in range(len(mean)):
    if i == len(mean)-1:
        x = TTran
    else:
        x = np.linspace(mean[i]-std[i], mean[i]+std[i], gridPts[i])
    InpArrays.append(x)

In [26]:
# Construct input data
Tinp = np.stack(
    np.meshgrid(*InpArrays, copy=False), axis=-1
).reshape(-1, len(InpArrays))
Binp = np.zeros((len(Tinp),5))
x = np.hstack([Binp,Tinp])
# Perform inference
TrunkPred = Trunk.predict(x)

In [27]:
def plot_modes(
    inp,
    data,
    title,
    fig_name
):
    # Set x- and y-axis -------------------------------------------------------
    x, y = inp[1], inp[0]
    # Limits
    xlim = [np.amin(x), np.amax(x)]
    ylim = [np.amin(y), np.amax(y)]

    # Initilize subplots ------------------------------------------------------
    shape = data.shape
    iMax, jMax = shape[2:4]
    fig, axs = plt.subplots(
        iMax, jMax, sharey=True, sharex=True, figsize=np.array([6.4,4.8])*2.5
    )
    axs = axs.reshape(iMax, jMax)

    # Subplots labeling -------------------------------------------------------
    cols = ['$r_{\\text{i}_i}-r_{\\text{i}_j}=\\,$'\
        +'${:.3f}$'.format(i) for i in inp[2]]
    rows = ['$\log_{10}\\left(r_{\\text{o}_i}/r_{\\text{o}_j}\\right)=\\,$'\
        +'${:.3f}$'.format(i) for i in inp[3]]
    # Column-wise
    for ax, col in zip(axs[0], cols):
        ax.set_title(col, pad=30, fontsize=20)
    # Row-wise
    for ax, row in zip(axs[:,-1], rows):
        ax.yaxis.set_label_position("right")
        ax.set_ylabel(row, rotation=-90, labelpad=60, fontsize=20)

    # Plot modes --------------------------------------------------------------
    for i in range(iMax):
        for j in range(jMax):
            axs[i,j].contourf(
                x,
                y,
                data[:,:,i,j],
                levels=20
            )
    fig.add_subplot(111, frame_on=False)
    plt.tick_params(labelcolor="none", bottom=False, left=False)
    plt.xticks([])
    plt.yticks([])
    plt.title(title, pad=90, fontsize=40)
    plt.xlabel(
        '$\log_{10}\\left(E_{\\text{vib}_i}/E_{\\text{vib}_j}\\right)$',
        labelpad=50
    )
    plt.ylabel(
        '$\log_{10}\\left(E_{\\text{rot}_i}/E_{\\text{rot}_j}\\right)$',
        labelpad=70
    )

    fig.savefig(fig_name, dpi=300)

In [28]:
# Plot modes
for m in tqdm(range(TrunkPred.shape[1])):
    path = ModesOutputFldr + '/Phi'+str(m+1).zfill(2)
    try:
        os.makedirs(path)
    except:
        pass
    phi = TrunkPred[:,m].reshape(gridPts)
    phi = np.split(phi, phi.shape[-1], axis=-1)
    for i, phi_i in enumerate(phi):
        plot_modes(
            inp=InpArrays,
            data=phi_i.squeeze(),
            title='Mode $\Phi_{{%s}}$ for $T={%s}$ K' % (m+1, int(TTran[i])),
            fig_name=path + f'/T{int(TTran[i])}.pdf'
        )

  0%|          | 0/64 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>